# Onset-HFO 5 — Did the HFO map point at the tissue whose removal cured the patient?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berdakh/onset-hfo/blob/master/onset-hfo/notebooks/05_surgical_outcome_study.ipynb)

Every other notebook here compares an algorithm to another algorithm, or to a
human reading the same screen. This one compares it to **what happened to the
patient after surgery** — the only reference standard in epilepsy surgery that
is not another opinion.

**The dataset makes it possible.** OpenNeuro `ds003498` (Zurich, 20 patients,
interictal slow-wave sleep, 2000 Hz) ships three things in the same archive:
expert HFO markings per channel, the **resected contacts** for each patient,
and whether that patient became **seizure-free**. Almost no public iEEG
dataset carries all three.

**What you will do.** Test a published finding on whole recordings, watch our
own detector land close behind the expert markings, and then see the same
analysis on the first 60 seconds give a *stronger* answer than the full five
minutes — which is the most useful result in the notebook, because it says the
number is not stable yet.

**Runtime.** About 45 minutes on a free Colab instance, most of it downloading
~2.2 GB (every run is exactly 300 s, so this is the whole recording and every
expert marking in it). Everything is cached, so a second run is compute-only.

In [1]:
# Colab setup. On your own machine, skip this cell and run
#   pip install -e ".[dev]"  from the onset-hfo directory instead.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/berdakh/onset-hfo.git"
BRANCH = "master"   # change to the default branch once this work is merged

if IN_COLAB and not os.path.exists("onset-hfo"):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1", REPO], check=True)
if IN_COLAB:
    os.chdir("/content/onset-hfo/onset-hfo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working directory:", os.getcwd())

working directory: /home/user/bci-gan/onset-hfo


---
## 1. The three ingredients

Read them straight out of the archive before running anything, so you can see
what the study is actually made of.

In [2]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from onset_hfo.clinical import fetch_participants, resection_map

participants = fetch_participants("ds003498")
print(participants[["subject", "epilepsy", "ilae", "outcome",
                    "months_follow_up"]].to_string(index=False))
print("\nseizure-free (S) vs recurrence (F):",
      participants["outcome"].value_counts().to_dict())

subject epilepsy  ilae outcome  months_follow_up
 sub-01      TLE     1       S                12
 sub-02      TLE     1       S                29
 sub-03      TLE     1       S                13
 sub-04      TLE     1       S                41
 sub-05      TLE     1       S                14
 sub-06      TLE     1       S                11
 sub-07      TLE     3       F                42
 sub-08      TLE     3       F                15
 sub-09      TLE     5       F                46
 sub-10      ETE     1       S                36
 sub-11      ETE     1       S                37
 sub-12      ETE     1       S                25
 sub-13      ETE     1       S                25
 sub-14      ETE     1       S                10
 sub-15      ETE     1       S                25
 sub-16      ETE     1       S                19
 sub-17      ETE     5       F                45
 sub-18      ETE     5       F                30
 sub-19      ETE     6       F                11
 sub-20      ETE    

`S` means **success** — seizure free. `F` means **failure** — seizures
returned. Reading `F` as "free" inverts every label in the cohort, which is
why `onset_hfo.clinical` never exposes the raw letter without the mapping
beside it.

Now the resected zone. The archive stores it as a clinician's free text, one
row per patient:

In [3]:
resections = resection_map("ds003498")
for subject in ["sub-01", "sub-12", "sub-15"]:
    r = resections[subject]
    print(f"{subject}: rz text {r.rz_text!r}")
    print(f"          -> {len(r.resected)} contacts: {', '.join(r.resected[:8])}"
          + (" ..." if len(r.resected) > 8 else ""))
    if r.eloquent:
        print(f"          eloquent (excluded by the source study): {', '.join(r.eloquent)}")
    if r.unparsed:
        print(f"          COULD NOT PARSE: {r.unparsed}  <- reported, not silently dropped")

[onset-hfo] sub-15: could not parse '1ll22-24' in the clinical sheet (left out of the contact lists)
sub-01: rz text 'ahr1-4, ar1-4, phr1-4'
          -> 12 contacts: AHR1, AHR2, AHR3, AHR4, AR1, AR2, AR3, AR4 ...
sub-12: rz text 'gl1-3, gl9-14, gl17-22, gl25-32, tl1-4'
          -> 27 contacts: GL1, GL2, GL3, GL9, GL10, GL11, GL12, GL13 ...
sub-15: rz text 'tbal2-4, tll1-2, tll9-10'
          -> 7 contacts: TBAL2, TBAL3, TBAL4, TLL1, TLL2, TLL9, TLL10
          eloquent (excluded by the source study): TLL6, TLL7, TLL8, TLL14, TLL15, TLL16, TLL31, TLL32
          COULD NOT PARSE: ('1ll22-24',)  <- reported, not silently dropped


That last line is the point of the `unparsed` field. Subject 15's sheet says
`1ll22-24` where every other token on the row says `tll` — a typo in the
original clinical data. A parser that quietly returned the other three ranges
would shrink that patient's excluded set with nothing to show for it.

## 2. Contacts are not channels

The sheet names **contacts** (`AHR1`). The analysis runs on **bipolar
channels** (`AHR1-AHR2`), each spanning two contacts. So a channel is inside
the resection, outside it, or — at the margin — straddling it:

In [4]:
from onset_hfo.clinical import classify_channels
from onset_hfo.datasets import fetch_slice
from onset_hfo.preprocess import prepare

rec = fetch_slice(dataset="ds003498", subject="sub-01", run="01",
                  t_start=0, t_stop=60, verbose=False)
prep = prepare(rec, verbose=False)
labels = classify_channels(prep.ch_names, resections["sub-01"])

print(labels["zone"].value_counts().to_string())
print("\nthe resection margin (one contact in, one out):")
print(labels[labels["zone"] == "partial"][["channel", "contact_a", "contact_b"]]
      .to_string(index=False))
print("\nhow much of the resection this recording can even see:")
print(resections["sub-01"].coverage(prep.ch_names))

zone
spared      31
resected     9
partial      3

the resection margin (one contact in, one out):
  channel contact_a contact_b
AHR4-AHR5      AHR4      AHR5
  AR4-AR5       AR4       AR5
PHR4-PHR5      PHR4      PHR5

how much of the resection this recording can even see:
{'subject': 'sub-01', 'rz_contacts_listed': 12, 'rz_contacts_recorded': 12, 'rz_coverage': 1.0, 'missing': ''}


**`partial` is kept as its own label on purpose.** Folding margin channels
into "resected" inflates every "we got it all" number; folding them into
"spared" inflates the opposite one. In the primary metric they sit in the
denominator and not the numerator, so a detector firing along the margin gets
no credit for it.

**`rz_coverage` is the honest denominator.** In five of the twenty patients —
all temporal-lobe cases, where the source study kept only the three most
mesial bipolar channels — only 4 of 16 resected contacts were recorded at all.
Their "share inside the resection" describes a quarter of a resection, and the
study writes that number to `recordings.csv` rather than burying it.

## 3. The design, before any numbers

Four choices, each of which can only make the result *worse*:

| Choice | Why |
|---|---|
| **An expert positive control** | Every number is computed twice — from the published expert markings and from our detector, on the same channels. Without it, a null is unreadable: it could be the detector or it could be 20 patients. Section 7 shows this control earning its keep. |
| **Operating points fixed in advance** | 2.0 SD for ripples, 5.0 SD for fast ripples, both chosen in `benchmark.py` on *channel-rank agreement with the experts* — a question that says nothing about surgery. Tuning a threshold against outcome and then reporting the outcome would be circular. |
| **Margin channels not claimed** | as above. |
| **Both channel scopes reported** | `reviewed` (what the annotators marked) and `all` (what a deployed tool would face). Reporting only the flattering one is a choice made after seeing both. |

## 4. Run it

In [5]:
from onset_hfo.outcome import outcome_study

# The default window is the whole run (300 s). ~2.2 GB cold, ~25 minutes cached.
result = outcome_study(verbose=True)
result.save()

[onset-hfo] sub-15: could not parse '1ll22-24' in the clinical sheet (left out of the contact lists)



[onset-hfo] (1/20) sub-01 [outcome S]
[onset-hfo] using cached slice /home/user/bci-gan/onset-hfo/artifacts/data/ds003498/sub-01_run-01_0-300s


[onset-hfo] loaded sub-01/None/run-01: 50 channels, 300.0 s @ 2000 Hz; seizure none marked


[onset-hfo]        expert: 0.32 of 3779 events inside the resection (9/23 reviewed channels resected)
[onset-hfo]           rms: 0.48 of 4627 events inside the resection (9/23 reviewed channels resected)

[onset-hfo] (2/20) sub-02 [outcome S]
[onset-hfo] using cached slice /home/user/bci-gan/onset-hfo/artifacts/data/ds003498/sub-02_run-01_0-300s


[onset-hfo] loaded sub-02/None/run-01: 64 channels, 300.0 s @ 2000 Hz; seizure none marked


...

[onset-hfo]        expert: 0.12 of 11871 events inside the resection (6/42 reviewed channels resected)
[onset-hfo]           rms: 0.11 of 11351 events inside the resection (6/42 reviewed channels resected)

[onset-hfo] (20/20) sub-20 [outcome F]
[onset-hfo] using cached slice /home/user/bci-gan/onset-hfo/artifacts/data/ds003498/sub-20_run-01_0-300s


[onset-hfo] loaded sub-20/None/run-01: 16 channels, 300.0 s @ 2000 Hz; seizure none marked


[onset-hfo]        expert: 0.29 of 4361 events inside the resection (5/12 reviewed channels resected)
[onset-hfo]           rms: 0.38 of 3062 events inside the resection (5/12 reviewed channels resected)


[onset-hfo] outcome study written to /home/user/bci-gan/onset-hfo/artifacts/results/outcome_ds003498


PosixPath('/home/user/bci-gan/onset-hfo/artifacts/results/outcome_ds003498')

## 5. The metric that carries the signal

Four metrics were computed. They disagree, and the disagreement is itself a
finding.

In [6]:
pd.set_option("display.width", 220)
print("--- was the single busiest channel resected? ---")
print(result.summary("top_channel_resected")
      .query("scope == 'reviewed'").to_string(index=False))
print("\n--- what fraction of all HFO events was inside the resection? ---")
print(result.summary("share_in_rz")
      .query("scope == 'reviewed'").to_string(index=False))

--- was the single busiest channel resected? ---
source    scope        band  n_seizure_free  n_recurrence  median_seizure_free  median_recurrence  mean_seizure_free  mean_recurrence   auc  auc_lo  auc_hi  rank_biserial  p_permutation  p_bonferroni
expert reviewed fast_ripple              13             7                  1.0                0.0              0.846            0.429 0.709   0.495   0.923          0.418          0.122           1.0
expert reviewed      ripple              13             7                  0.0                0.0              0.308            0.286 0.511   0.291   0.731          0.022          1.000           1.0
   rms reviewed fast_ripple              13             7                  1.0                0.0              0.769            0.429 0.670   0.451   0.890          0.341          0.174           1.0
   rms reviewed      ripple              13             7                  1.0                0.0              0.615            0.143 0.736   0.549   0

Read the `expert` / `fast_ripple` row of the first table first. The busiest
fast-ripple channel was inside the resection in **11 of 13** patients who
became seizure-free and **3 of 7** whose seizures returned — AUC 0.71,
permutation p = 0.12. The direction is the one Fedele et al. 2017 predicts, and
with 13 patients against 7 it does not reach significance. Nothing in this
study does.

Then read the same row of the second table: `share_in_rz` shows **nothing** in
the expert arm — AUC 0.48, a coin flip — on the same events. **Concentration
localises; proportion does not.** "Most of this patient's HFOs were inside the
resection" is largely a statement about how big the resection was. "The one
place generating the most fast ripples was removed" is the clinically useful
sentence — and it means any report built on this pipeline should show a
*ranking*, not a percentage. This is the one conclusion in the notebook that
holds at every window length.

## 6. Where our detector stands

In [7]:
print(result.verdict())          # the pre-specified comparison
print()
for key, text in result.verdicts().items():
    if key.startswith("top_channel"):
        print(f"{key}:\n  {text}\n")

expert: AUC 0.71 (95% CI 0.49-0.92), p = 0.122 (24-comparison Bonferroni 1.00) -- does not separate the groups. rms: AUC 0.67 (95% CI 0.45-0.89), p = 0.174 (24-comparison Bonferroni 1.00) -- does not separate the groups. With 13 seizure-free and 7 recurrence patients, the smallest effect reaching 80% power is AUC 0.85; anything below that could not have been detected here whether or not it is real. The expert markings do not separate the groups either, so this is a statement about a 60-second window and 20 patients, not about the detector.

top_channel_resected/fast_ripple:
  expert: AUC 0.71 (95% CI 0.49-0.92), p = 0.122 (24-comparison Bonferroni 1.00) -- does not separate the groups. rms: AUC 0.67 (95% CI 0.45-0.89), p = 0.174 (24-comparison Bonferroni 1.00) -- does not separate the groups. With 13 seizure-free and 7 recurrence patients, the smallest effect reaching 80% power is AUC 0.85; anything below that could not have been detected here whether or not it is real. The expert mark

In [8]:
subjects = result.subjects.merge(result.participants[["subject", "outcome"]], on="subject")
view = subjects.query("band == 'fast_ripple' and scope == 'reviewed'")
print(view[["subject", "outcome", "source", "n_events",
            "share_in_rz", "top_channel_resected"]]
      .sort_values(["outcome", "subject", "source"]).to_string(index=False))

subject outcome source  n_events  share_in_rz  top_channel_resected
 sub-07       F expert     116.0     0.413793                   0.0
 sub-07       F    rms       9.0     0.222222                   0.0
 sub-08       F expert     105.0     0.447619                   0.0
 sub-08       F    rms     245.0     0.000000                   0.0
 sub-09       F expert    1355.0     0.760886                   1.0
 sub-09       F    rms     661.0     0.888048                   1.0
 sub-17       F expert    2448.0     0.787173                   1.0
 sub-17       F    rms    2180.0     0.784862                   1.0
 sub-18       F expert     876.0     0.390411                   1.0
 sub-18       F    rms     257.0     0.778210                   1.0
 sub-19       F expert    3162.0     0.161290                   0.0
 sub-19       F    rms    1537.0     0.202993                   0.0
 sub-20       F expert     382.0     0.285340                   0.0
 sub-20       F    rms     110.0     0.163636   

**Our detector lands just behind the expert markings** — 10 of 13 versus 3 of
7, AUC 0.67 against their 0.71, with intervals that overlap almost entirely.
Both are null. Over 300 s every subject produces fast-ripple detections, so
unlike the 60-second version of this analysis no patient drops out.

Keep that 0.67-against-0.71 in mind for the next section, because on the first
minute of the same recordings the two numbers were 0.70 and **0.82**.

## 7. The window changes the answer

This is the most important cell in the notebook. Same code, same pre-specified
metric, same patients — only the analysis window differs.

In [9]:
short = outcome_study(t_stop=60.0, bands=("fast_ripple",), verbose=False)
for label, res in [("first 60 s", short), ("whole 300 s", result)]:
    row = res.summary("top_channel_resected").query(
        "scope == 'reviewed' and band == 'fast_ripple'")
    print(f"--- {label} ---")
    print(row[["source", "mean_seizure_free", "mean_recurrence",
               "auc", "auc_lo", "auc_hi", "p_permutation"]].to_string(index=False))

--- first 60 s ---
source  mean_seizure_free  mean_recurrence   auc  auc_lo  auc_hi  p_permutation
expert              0.923            0.286 0.819   0.637   1.000          0.007
   rms              0.833            0.429 0.702   0.488   0.917          0.129
--- whole 300 s ---
source  mean_seizure_free  mean_recurrence   auc  auc_lo  auc_hi  p_permutation
expert              0.846            0.429 0.709   0.495   0.923          0.122
   rms              0.769            0.429 0.670   0.451   0.890          0.174


In [10]:
# Which patients moved, and in which direction?
key = ["subject", "source"]
q = "band == 'fast_ripple' and scope == 'reviewed'"
moved = (short.subjects.query(q)[key + ["n_events", "top_channel_resected"]]
         .merge(result.subjects.query(q)[key + ["n_events", "top_channel_resected"]],
                on=key, suffixes=("_60", "_300"))
         .merge(result.participants[["subject", "outcome"]], on="subject"))
print(moved[moved["top_channel_resected_60"] != moved["top_channel_resected_300"]]
      .to_string(index=False))

subject source  n_events_60  top_channel_resected_60  n_events_300  top_channel_resected_300 outcome
 sub-10    rms          0.0                      NaN          15.0                       1.0       S
 sub-14    rms         30.0                      1.0         205.0                       0.0       S
 sub-15 expert        128.0                      1.0         592.0                       0.0       S
 sub-18 expert        166.0                      0.0         876.0                       1.0       F


**Five times the data, a weaker result.** The expert arm falls from AUC 0.82
(p = 0.007) to 0.71 (p = 0.12); ours barely moves, 0.70 to 0.67. Two patients
account for all of it: `sub-18` (a recurrence whose busiest channel turns out
to be *inside* the resection once you look past the first minute — a flip that
costs twice) and `sub-15` (a seizure-free patient going the other way).

Three things follow, and they are why this notebook exists:

- **The 60-second number should never have been the headline**, and it was:
  the project's README and landing page carried AUC 0.82, p = 0.007 until this
  was run. 60 s was chosen for download size before any outcome data was
  touched, and the metric and band were pre-specified — so this is not
  cherry-picking. It is something worse and more common: an analysis window
  short enough to change the conclusion, never checked.
- **"Which channel is busiest" is fragile.** It is an argmax over 6–65 channels
  whose Poisson rate intervals overlap. `metrics.py` already refuses to rank
  channels whose intervals overlap when it reports rates; this metric does not,
  and it should.
- **The positive control earned its place** — just not in the way it was
  designed to. It was built to tell "our detector is worse" apart from "this
  study is underpowered". The 60-second run looked like the first. The full run
  says the second.

## 8. Why the band matters, demonstrated

One more parameter that decides whether there is a signal at all. The first
version of this analysis used 2.0 SD in **both** bands, because that is what
the ripple benchmark prefers. Here is what that does in the fast-ripple band:

In [11]:
wrong = outcome_study(threshold_sd=2.0,   # one threshold everywhere
                      bands=("fast_ripple",), verbose=False)
print("2.0 SD in both bands:")
print(wrong.summary("top_channel_resected").query("scope == 'reviewed'").to_string(index=False))
print("\nmeasured per-band operating points:")
print(result.summary("top_channel_resected")
      .query("scope == 'reviewed' and band == 'fast_ripple'").to_string(index=False))

2.0 SD in both bands:
source    scope        band  n_seizure_free  n_recurrence  median_seizure_free  median_recurrence  mean_seizure_free  mean_recurrence   auc  auc_lo  auc_hi  rank_biserial  p_permutation  p_bonferroni
expert reviewed fast_ripple              13             7                  1.0                0.0              0.846            0.429 0.709   0.495   0.923          0.418          0.122         1.000
   rms reviewed fast_ripple              13             7                  1.0                0.0              0.846            0.286 0.780   0.566   0.962          0.560          0.022         0.268

measured per-band operating points:
source    scope        band  n_seizure_free  n_recurrence  median_seizure_free  median_recurrence  mean_seizure_free  mean_recurrence   auc  auc_lo  auc_hi  rank_biserial  p_permutation  p_bonferroni
expert reviewed fast_ripple              13             7                  1.0                0.0              0.846            0.429 0.709  

At 2.0 SD the fast-ripple detector runs at precision 0.086 — a mean of 1,142
detections per 60 s against a mean of 228 expert-marked events. Nothing about
the outcome data was used to choose either threshold; both come from
channel-rank agreement with the experts, in
`notebooks/03_validation_and_benchmark.ipynb`. **One threshold for both bands
is a bug, not a simplification.**

## 9. Read this before quoting any of it

- **Thirteen versus seven is a very small study.** `min_detectable_auc(13, 7)`
  returns **0.85** — with these group sizes only a very large separation
  reaches 80% power. A p above 0.05 here means *underpowered*, not *no effect*.
- **Nothing in the full-run study reaches p < 0.05**, and the Bonferroni column
  in `groups.csv` corrects across 24 comparisons. Read every row as
  hypothesis-generating.
- **One full run of one night**, against several whole nights in the source
  study. The archive has 1–6 runs per subject; combining them is the obvious
  extension, and given §7 it should happen before any number here is called
  stable.
- **Retrospective, one centre, one surgical team.** The
  [HFO Trial](https://www.thelancet.com/journals/laneur/article/PIIS1474-4422(22)00311-8/fulltext)
  (Lancet Neurology 2022) tested HFO-guided resection prospectively and did not
  find the benefit retrospective series report. This is a retrospective series.
- **Reproducing a retrospective result is evidence that the analysis is sound,
  not that the clinical claim is.**

`docs/OUTCOME.md` carries the full design, every table, and the list of what
this cannot support.

In [12]:
from onset_hfo.outcome import min_detectable_auc
print("smallest AUC detectable at 80% power with 13 vs 7:",
      min_detectable_auc(13, 7))
print("\nfiles written:")
import pathlib
for f in sorted(pathlib.Path("artifacts/results/outcome_ds003498").iterdir()):
    print("  ", f.name)

smallest AUC detectable at 80% power with 13 vs 7: 0.85

files written:
   channels.csv
   groups.csv
   participants.csv
   recordings.csv
   run.json
   subjects.csv
